In [1]:
import numpy as np
import pandas as pd

# --- Project & Financial Inputs ---
CAPEX = 50_000_000         # Initial Capital Expenditure (€50M)
PROJECT_LIFETIME = 15      # Years
DISCOUNT_RATE = 0.08       # Weighted Average Cost of Capital (WACC/Discount Rate)
DAILY_CYCLES_TARGET = 1.5  # Strategic goal: 1.5 equivalent full cycles per day
MAX_CYCLES_SPEC = 5000     # Warranty limit (determines degradation sensitivity)

In [3]:
# --- 1. Strategy Model (Inputs from Operational/Degradation decisions) ---

def model_annual_revenue(daily_profit_base, target_cycles, capacity, start_year=1):
    """
    Simulates the declining annual revenue based on capacity degradation.
    Assumption: Revenue is proportional to available capacity (SoH).
    Degradation: Simplified linear loss based on cumulative cycles (a key strategic risk).
    """
    
    annual_revenues = {}
    current_soh = 1.0 # Starts at 100%
    DAYS_PER_YEAR = 365
    
    # Calculate lifetime cycles consumed by strategy
    annual_cycles_consumed = target_cycles * DAYS_PER_YEAR
    
    for year in range(start_year, PROJECT_LIFETIME + 1):
        
        # 1. Capacity Loss Model (Degradation)
        # Use a simplified linear degradation based on total cycles vs warranty limit
        # This penalizes high cycling (high DoD)
        soh_loss_rate = annual_cycles_consumed / MAX_CYCLES_SPEC
        current_soh = np.maximum(0.80, current_soh - soh_loss_rate) # Capacity never falls below 80%

        # 2. Revenue Calculation
        # Annual revenue declines with capacity loss
        annual_profit = daily_profit_base * DAYS_PER_YEAR * current_soh
        annual_revenues[year] = annual_profit
        
    return annual_revenues, current_soh


In [4]:
# --- 2. Valuation Model (Discounted Cash Flow - DCF) ---

def calculate_dcf_metrics(annual_cash_flows, initial_capex, discount_rate):
    """Calculates Net Present Value (NPV) and Internal Rate of Return (IRR)."""
    
    # NPV Calculation
    npv = -initial_capex
    for year, cash_flow in annual_cash_flows.items():
        discount_factor = 1 / (1 + discount_rate)**year
        npv += cash_flow * discount_factor
        
    # IRR Calculation (Requires a list of all cash flows, including initial CAPEX)
    # Note: Requires a complete cash flow list for NumPy's solver
    cf_list = [-initial_capex] + list(annual_cash_flows.values())
    try:
        irr = np.irr(cf_list)
    except:
        irr = np.nan
        
    return npv, irr

In [5]:
# 1. Sample Strategic Input (e.g., from Stochastic Optimization)
# Assume the optimal *risk-averse* dispatch yields a gross profit of 
# €80,000 per day IF the battery were 100% efficient and new.
BASE_DAILY_PROFIT_NEW = 80000 
BESS_MWh = 100 # Assuming 100 MWh BESS

# 2. Run Strategy Model (Generate Revenue Stream based on degradation)
annual_revenues, final_soh = model_annual_revenue(
    BASE_DAILY_PROFIT_NEW, DAILY_CYCLES_TARGET, BESS_MWh
)

# 3. Run Valuation Model (DCF)
npv, irr = calculate_dcf_metrics(annual_revenues, CAPEX, DISCOUNT_RATE)

In [ ]:
# 4. Results Reporting
print("--- Asset Strategy & Valuation Results ---")
print(f"BESS Capacity: {BESS_MWh} MWh | Strategic Cycles/Day: {DAILY_CYCLES_TARGET}")
print(f"Project Lifetime: {PROJECT_LIFETIME} years | Discount Rate: {DISCOUNT_RATE*100:.1f}%")
print("-" * 50)
print(f"Final State of Health (SoH): {final_soh*100:.1f}%")
print(f"Initial Investment (CAPEX): {CAPEX:,.0f} €")
print(f"Calculated Net Present Value (NPV): {npv:,.0f} €")
print(f"Calculated Internal Rate of Return (IRR): {irr*100:.2f}%")
print("-" * 50)

# Strategic Conclusion (The 'Why' for the Newbie)
if npv > 0:
    print(f"\nSTRATEGY CONCLUSION: The NPV is positive, meaning the asset's strategy is financially sound. The project is expected to generate {npv:,.0f} € more than the cost of capital, justifying the investment.")
else:
    print("\nSTRATEGY CONCLUSION: The NPV is negative. The current operational strategy (or high initial cost) makes the project unprofitable over its lifetime.")